[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/29.%20The%20Integrated%20Quay%20Crane%20%26%20Yard%20Truck%20Scheduling%20Problem/P29-Tier-4_executed.ipynb)

---


# 29. The Integrated Quay Crane and Yard Truck Scheduling Problem

## Tier 4: Clustering-Augmented Scheduling

### Goal

This notebook demonstrates a two-stage scheduling heuristic. First, K-means groups containers with similar measured characteristics. Next, interpretable rules assign a scheduling strategy to each group, and the containers are scheduled on crane–truck pairs. Clustering is unsupervised: it discovers groups from feature values without using known group labels. It does not learn which schedule is best.

### Inputs and outputs

The input is a set of containers with handling, distance, urgency, weight, accessibility, vessel-position, and yard-block attributes, together with a set of crane–truck pairs. The output is a schedule containing one assignment per container, plus descriptive profiles for the clusters. Assignment times are expressed in minutes under the simplified processing-time model defined below.

### Scope and modeling assumptions

- The example contains 12 containers and uses three K-means clusters. These are the supplied demonstration data and parameter values, not a claim that three groups are best for every terminal.
- Each crane–truck pair is treated as an independent scheduling resource. The example includes pairs that share a crane or truck; the model does not enforce separate crane and truck non-overlap constraints. Consequently, the schedule is illustrative under the pair-resource abstraction and is not a complete physical terminal schedule.
- Each assignment's processing duration is modeled as handling time plus half the yard distance. This is a simplifying assumption, not a travel-time model calibrated to a particular terminal.
- Deadlines, release times, precedence constraints, crane interference, truck routing, and uncertainty are not modeled. Cluster-specific rules are heuristics; neither the clustering nor the resulting schedule is guaranteed to be optimal.
- The feature `yard_block` is used by the batch-processing rule, but is not one of the six features used to form clusters.

### Method at a glance

1. Standardize six numeric features so that features with larger measurement scales do not dominate Euclidean distance solely because of their units.
2. Apply K-means with the configured number of clusters and random seed.
3. Summarize each cluster in the original feature units and choose a rule-based strategy from its average handling time, distance, and urgency.
4. Process clusters in descending average urgency, scheduling their containers with the selected strategy and updating pair availability.
5. Inspect cluster profiles, schedule visualizations, descriptive feature distributions, and the included simple baseline comparison.

### Mathematical model

Let \(i \in \{1,\ldots,n\}\) index containers, let \(x_i \in \mathbb{R}^{6}\) be the feature vector used for clustering, and let \(z_i \in \mathbb{R}^{6}\) be its standardized version. K-means assigns each container to one of \(K\) clusters by minimizing the within-cluster sum of squared Euclidean distances:

$$
\min_{c_1,\ldots,c_n,\,\mu_1,\ldots,\mu_K}
\sum_{i=1}^{n} \left\|z_i-\mu_{c_i}\right\|_2^2,
\qquad c_i \in \{0,\ldots,K-1\}.
$$

Here, \(c_i\) is the cluster label of container \(i\), and \(\mu_k\) is the standardized-space centroid of cluster \(k\). Standardization uses each feature's sample mean and standard deviation, as implemented by `StandardScaler`. K-means alternates between assigning each point to its nearest centroid and updating each centroid to the mean of its assigned points; `KMeans.fit_predict` performs this procedure. The inertia is the minimized sum of squared distances for the fitted labels. The silhouette score is a separate clustering diagnostic: it compares each point's within-cluster distance with its distance to the nearest other cluster. Neither metric establishes that the resulting schedules are operationally effective.

After clustering, a rule maps each cluster profile to one of four strategies. For a container assigned to pair \(p\), the simplified duration and timing recurrence are:

$$
d_i = h_i + \frac{r_i}{2}, \qquad
s_i = a_p, \qquad
C_i = s_i+d_i, \qquad
a_p \leftarrow C_i.
$$

Here \(h_i\) is `handling_time`, \(r_i\) is `yard_distance`, \(a_p\) is the pair's `available_time` before the assignment, \(s_i\) is its start time, and \(C_i\) is its completion time. The update makes the next assignment to that pair start no earlier than its preceding completion. The cluster strategy determines container order and pair selection; the example does not solve a global makespan-minimization problem.

For \(n\) containers, \(K\) clusters, \(d=6\) features, and \(I\) K-means iterations, clustering costs approximately \(O(I n K d)\). The scheduling heuristics process containers individually; their pair-selection work is linear in the number of pairs per container. This computational description does not imply an optimality guarantee.

### Interpreting the results

Cluster numbers are arbitrary labels: for example, cluster 0 has no inherent meaning beyond the group assigned that label in this fit. Read the profile statistics and recommended strategy to understand a group. The silhouette score measures separation in standardized feature space, not schedule quality. Feature variability statistics and plots describe the data; K-means does not provide a feature-importance ranking. The baseline comparison reports results for the included heuristic schedules only, and a single randomized baseline run is not a statistical comparison.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

plt.style.use("default")
sns.set_palette("husl")

# Fixing the seed makes the randomized parts of the example reproducible.
np.random.seed(42)

In [ ]:
@dataclass
class Container:
    """Container attributes used for clustering and scheduling."""

    id: int
    handling_time: float
    yard_distance: float
    deadline_urgency: float  # Scale 0–1; larger values indicate greater urgency.
    weight_class: int  # Scale 1–5; larger values indicate heavier containers.
    crane_accessibility: float  # Scale 0–1; larger values indicate easier access.
    vessel_bay_position: int  # Position along the vessel, from 1 to 20.
    yard_block: int  # Yard block, from 1 to 4.


@dataclass
class CraneTruckPair:
    """A crane–truck pair treated as one scheduling resource."""

    crane_id: int
    truck_id: int
    available_time: float = 0.0
    total_processing: float = 0.0


@dataclass
class ClusterProfile:
    """Descriptive statistics and the rule-selected strategy for a cluster."""

    cluster_id: int
    avg_handling_time: float
    avg_yard_distance: float
    avg_urgency: float
    avg_weight_class: float
    avg_accessibility: float
    container_count: int
    recommended_strategy: str


@dataclass
class Assignment:
    """Scheduled container assignment to a crane–truck pair."""

    container_id: int
    crane_id: int
    truck_id: int
    start_time: float
    completion_time: float
    cluster_strategy: str
    cluster_id: int

In [ ]:
class ClusterBasedScheduler:
    """Cluster containers, profile groups, and schedule with rule-based strategies."""

    def __init__(self, n_clusters: int = 3, random_state: int = 42):
        if n_clusters < 2:
            raise ValueError("n_clusters must be at least 2 to calculate a silhouette score.")

        self.n_clusters = n_clusters
        self.random_state = random_state
        self.kmeans = KMeans(
            n_clusters=n_clusters,
            random_state=random_state,
            n_init=10,
        )
        self.scaler = StandardScaler()
        self.cluster_profiles: Dict[int, ClusterProfile] = {}
        self.container_cluster_ids: Dict[int, int] = {}
        self.feature_names = [
            "handling_time",
            "yard_distance",
            "deadline_urgency",
            "weight_class",
            "crane_accessibility",
            "vessel_bay_position",
        ]

    def extract_features(self, containers: List[Container]) -> np.ndarray:
        """Build the six-dimensional feature vector used by K-means."""
        return np.asarray(
            [
                [
                    container.handling_time,
                    container.yard_distance,
                    container.deadline_urgency,
                    container.weight_class,
                    container.crane_accessibility,
                    container.vessel_bay_position,
                ]
                for container in containers
            ],
            dtype=float,
        )

    def cluster_containers(
        self, containers: List[Container]
    ) -> Dict[int, List[Container]]:
        """Standardize features, fit K-means, and group containers by label."""
        if len(containers) <= self.n_clusters:
            raise ValueError("The number of containers must exceed n_clusters.")
        if len({container.id for container in containers}) != len(containers):
            raise ValueError("Container IDs must be unique.")

        features = self.extract_features(containers)
        if not np.isfinite(features).all():
            raise ValueError("Clustering features must all be finite numeric values.")

        features_normalized = self.scaler.fit_transform(features)
        cluster_labels = self.kmeans.fit_predict(features_normalized)
        if len(np.unique(cluster_labels)) < 2:
            raise ValueError("K-means produced fewer than two non-empty clusters.")

        clusters: Dict[int, List[Container]] = {}
        self.container_cluster_ids = {}
        for container, label in zip(containers, cluster_labels):
            cluster_id = int(label)
            clusters.setdefault(cluster_id, []).append(container)
            self.container_cluster_ids[container.id] = cluster_id

        # These diagnostics describe feature-space clustering, not schedule quality.
        self.silhouette_score = silhouette_score(features_normalized, cluster_labels)
        self.inertia = self.kmeans.inertia_
        return clusters

    def analyze_cluster_characteristics(
        self, clusters: Dict[int, List[Container]]
    ) -> Dict[int, ClusterProfile]:
        """Summarize cluster attributes and select a strategy using fixed rules."""
        profiles: Dict[int, ClusterProfile] = {}

        for cluster_id, cluster_containers in clusters.items():
            avg_handling = float(np.mean([c.handling_time for c in cluster_containers]))
            avg_distance = float(np.mean([c.yard_distance for c in cluster_containers]))
            avg_urgency = float(np.mean([c.deadline_urgency for c in cluster_containers]))
            avg_weight = float(np.mean([c.weight_class for c in cluster_containers]))
            avg_accessibility = float(
                np.mean([c.crane_accessibility for c in cluster_containers])
            )
            strategy = self._determine_strategy(
                avg_handling, avg_distance, avg_urgency
            )

            profiles[cluster_id] = ClusterProfile(
                cluster_id=cluster_id,
                avg_handling_time=avg_handling,
                avg_yard_distance=avg_distance,
                avg_urgency=avg_urgency,
                avg_weight_class=avg_weight,
                avg_accessibility=avg_accessibility,
                container_count=len(cluster_containers),
                recommended_strategy=strategy,
            )

        self.cluster_profiles = profiles
        return profiles

    def _determine_strategy(
        self, avg_handling: float, avg_distance: float, avg_urgency: float
    ) -> str:
        """Map cluster averages to the example's interpretable scheduling rules."""
        if avg_urgency > 0.8 and avg_handling < 3.0:
            return "priority_fast_track"
        if avg_distance > 6.0 and avg_handling > 4.0:
            return "batch_processing"
        if avg_urgency < 0.4:
            return "efficiency_optimization"
        return "balanced_allocation"

    def schedule_cluster(
        self,
        cluster_containers: List[Container],
        strategy: str,
        pairs: List[CraneTruckPair],
    ) -> List[Assignment]:
        """Schedule one cluster and update the selected pairs' availability."""
        if not pairs:
            raise ValueError("At least one crane–truck pair is required.")

        schedule: List[Assignment] = []

        if strategy == "priority_fast_track":
            ordered_containers = sorted(
                cluster_containers, key=lambda container: -container.deadline_urgency
            )
            for container in ordered_containers:
                pair = self._select_best_pair(container, pairs)
                assignment = self._create_assignment(container, pair, strategy)
                schedule.append(assignment)
                pair.available_time = assignment.completion_time

        elif strategy == "batch_processing":
            yard_groups: Dict[int, List[Container]] = {}
            for container in cluster_containers:
                yard_groups.setdefault(container.yard_block, []).append(container)

            for yard_block, yard_containers in yard_groups.items():
                for container in sorted(yard_containers, key=lambda item: item.handling_time):
                    pair = self._select_yard_optimized_pair(
                        container, pairs, yard_block
                    )
                    assignment = self._create_assignment(container, pair, strategy)
                    schedule.append(assignment)
                    pair.available_time = assignment.completion_time

        elif strategy == "efficiency_optimization":
            ordered_containers = sorted(
                cluster_containers, key=lambda container: container.handling_time
            )
            for container in ordered_containers:
                pair = self._select_efficient_pair(container, pairs)
                assignment = self._create_assignment(container, pair, strategy)
                schedule.append(assignment)
                pair.available_time = assignment.completion_time

        elif strategy == "balanced_allocation":
            for index, container in enumerate(cluster_containers):
                pair = pairs[index % len(pairs)]
                assignment = self._create_assignment(container, pair, strategy)
                schedule.append(assignment)
                pair.available_time = assignment.completion_time

        else:
            raise ValueError(f"Unknown scheduling strategy: {strategy}")

        return schedule

    def _select_best_pair(
        self, container: Container, pairs: List[CraneTruckPair]
    ) -> CraneTruckPair:
        """Choose the pair with the earliest predicted completion for this container."""
        return min(
            pairs,
            key=lambda pair: max(pair.available_time, 0.0)
            + container.handling_time
            + container.yard_distance / 2.0,
        )

    def _select_yard_optimized_pair(
        self,
        container: Container,
        pairs: List[CraneTruckPair],
        yard_block: int,
    ) -> CraneTruckPair:
        """Prefer a nearby truck ID, using availability to break location ties."""
        return min(
            pairs,
            key=lambda pair: (
                abs(pair.truck_id - yard_block),
                pair.available_time,
            ),
        )

    def _select_efficient_pair(
        self, container: Container, pairs: List[CraneTruckPair]
    ) -> CraneTruckPair:
        """Choose the earliest available pair under the container duration estimate."""
        return min(
            pairs,
            key=lambda pair: max(pair.available_time, 0.0)
            + container.handling_time
            + container.yard_distance / 2.0,
        )

    def _create_assignment(
        self, container: Container, pair: CraneTruckPair, strategy: str
    ) -> Assignment:
        """Create a timed assignment using the stated duration assumption."""
        start_time = max(pair.available_time, 0.0)
        duration = container.handling_time + container.yard_distance / 2.0
        completion_time = start_time + duration
        pair.total_processing += duration

        return Assignment(
            container_id=container.id,
            crane_id=pair.crane_id,
            truck_id=pair.truck_id,
            start_time=start_time,
            completion_time=completion_time,
            cluster_strategy=strategy,
            cluster_id=self._get_container_cluster_id(container),
        )

    def _get_container_cluster_id(self, container: Container) -> int:
        """Return the fitted cluster label associated with a container ID."""
        try:
            return self.container_cluster_ids[container.id]
        except KeyError as error:
            raise ValueError("Container must be clustered before it is scheduled.") from error

    def integrated_cluster_schedule(
        self,
        containers: List[Container],
        pairs: List[CraneTruckPair],
    ) -> Tuple[List[Assignment], Dict[int, ClusterProfile]]:
        """Cluster, profile, and schedule containers in the example's prescribed order."""
        if not pairs:
            raise ValueError("At least one crane–truck pair is required.")

        clusters = self.cluster_containers(containers)
        profiles = self.analyze_cluster_characteristics(clusters)
        complete_schedule: List[Assignment] = []

        # Processing high-average-urgency clusters first carries the stated priority rule.
        sorted_clusters = sorted(
            clusters.items(),
            key=lambda item: profiles[item[0]].avg_urgency,
            reverse=True,
        )
        for cluster_id, cluster_containers in sorted_clusters:
            strategy = profiles[cluster_id].recommended_strategy
            complete_schedule.extend(
                self.schedule_cluster(cluster_containers, strategy, pairs)
            )

        assigned_ids = [assignment.container_id for assignment in complete_schedule]
        expected_ids = [container.id for container in containers]
        if sorted(assigned_ids) != sorted(expected_ids):
            raise AssertionError("The schedule must assign every container exactly once.")

        return complete_schedule, profiles

In [ ]:
def create_concrete_example() -> Tuple[List[Container], List[CraneTruckPair]]:
    """Create the 12-container demonstration instance and its four pairs."""
    containers = [
        # Fast and urgent examples.
        Container(1, 2.1, 3.2, 0.90, 2, 0.80, 3, 1),
        Container(2, 1.8, 2.8, 0.85, 1, 0.90, 5, 1),
        Container(3, 2.3, 3.5, 0.88, 2, 0.85, 7, 2),
        Container(4, 1.9, 2.9, 0.92, 1, 0.95, 2, 1),
        # Medium examples.
        Container(5, 3.2, 4.5, 0.45, 3, 0.60, 8, 2),
        Container(6, 3.5, 5.1, 0.42, 3, 0.55, 11, 3),
        Container(7, 2.9, 4.8, 0.48, 2, 0.65, 9, 2),
        Container(8, 3.3, 5.3, 0.44, 3, 0.58, 12, 3),
        Container(9, 3.1, 4.7, 0.46, 2, 0.62, 10, 2),
        # Slow and far examples.
        Container(10, 4.8, 8.2, 0.25, 4, 0.30, 15, 4),
        Container(11, 5.1, 8.8, 0.22, 5, 0.25, 17, 4),
        Container(12, 4.6, 7.9, 0.28, 4, 0.35, 14, 3),
    ]

    pairs = [
        CraneTruckPair(crane_id=0, truck_id=0),
        CraneTruckPair(crane_id=0, truck_id=1),
        CraneTruckPair(crane_id=1, truck_id=0),
        CraneTruckPair(crane_id=1, truck_id=1),
    ]
    return containers, pairs


containers, pairs = create_concrete_example()
print(f"Problem created: {len(containers)} containers, {len(pairs)} crane–truck pairs")
print("\nSample container characteristics:")
for container in containers[:3]:
    print(
        f"Container {container.id}: handling={container.handling_time:.1f} min, "
        f"distance={container.yard_distance:.1f}, "
        f"urgency={container.deadline_urgency:.2f}"
    )

In [ ]:
def run_clustering_scheduler():
    """Fit the clustering model, create a schedule, and report elapsed time."""
    scheduler = ClusterBasedScheduler(n_clusters=3, random_state=42)
    print("=== CLUSTER-BASED SCHEDULING START ===")
    start_time = time.time()
    schedule, profiles = scheduler.integrated_cluster_schedule(containers, pairs)
    computation_time = time.time() - start_time
    print("\n=== CLUSTER-BASED SCHEDULING COMPLETE ===")
    return scheduler, schedule, profiles, computation_time


scheduler, schedule, profiles, computation_time = run_clustering_scheduler()

print(f"\nComputation time: {computation_time:.4f} seconds")
print(f"Clustering silhouette score: {scheduler.silhouette_score:.3f}")
print(f"K-means inertia: {scheduler.inertia:.3f}")
print(f"Total assignments: {len(schedule)}")

print("\n=== CLUSTER PROFILES ===")
for cluster_id, profile in sorted(profiles.items()):
    print(f"\nCluster {cluster_id} ({profile.recommended_strategy}):")
    print(f"  Containers: {profile.container_count}")
    print(f"  Average handling time: {profile.avg_handling_time:.2f} min")
    print(f"  Average yard distance: {profile.avg_yard_distance:.2f}")
    print(f"  Average urgency: {profile.avg_urgency:.2f}")

In [ ]:
def visualize_clustering_results(scheduler, schedule, profiles, computation_time):
    """Plot feature-space groups, strategy summaries, and the resulting schedule."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(
        "Clustering-Augmented Container Scheduling",
        fontsize=16,
        fontweight="bold",
    )

    # Show two original features; K-means itself uses all six standardized features.
    ax1 = axes[0, 0]
    features = scheduler.extract_features(containers)
    labels = scheduler.kmeans.labels_
    scatter = ax1.scatter(
        features[:, 0], features[:, 1], c=labels, cmap="viridis", s=100, alpha=0.75
    )
    centers = scheduler.scaler.inverse_transform(scheduler.kmeans.cluster_centers_)
    ax1.scatter(
        centers[:, 0], centers[:, 1], c="red", marker="x", s=200, linewidths=3,
        label="Centroids projected onto these two features",
    )
    ax1.set_xlabel("Handling time (minutes)")
    ax1.set_ylabel("Yard distance (model units)")
    ax1.set_title("Clusters in Two Original Features")
    ax1.legend(loc="best", fontsize=8)
    ax1.grid(True, alpha=0.3)
    fig.colorbar(scatter, ax=ax1, label="Cluster label")

    # These normalized profile values are display indices, not model inputs.
    ax2 = axes[0, 1]
    categories = ["Handling time", "Yard distance", "Urgency", "Weight", "Access"]
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    closed_angles = angles + angles[:1]
    for cluster_id, profile in sorted(profiles.items()):
        values = [
            profile.avg_handling_time / 5.0,
            profile.avg_yard_distance / 10.0,
            profile.avg_urgency,
            profile.avg_weight_class / 5.0,
            profile.avg_accessibility,
        ]
        closed_values = values + values[:1]
        ax2.plot(closed_angles, closed_values, "o-", linewidth=2, label=f"Cluster {cluster_id}")
        ax2.fill(closed_angles, closed_values, alpha=0.15)
    ax2.set_xticks(angles)
    ax2.set_xticklabels(categories)
    ax2.set_ylim(0, 1.1)
    ax2.set_title("Scaled Cluster Profile Summaries")
    ax2.legend(loc="best", fontsize=8)

    ax3 = axes[0, 2]
    strategy_counts: Dict[str, int] = {}
    for profile in profiles.values():
        label = profile.recommended_strategy.replace("_", " ").title()
        strategy_counts[label] = strategy_counts.get(label, 0) + profile.container_count
    ax3.pie(
        list(strategy_counts.values()),
        labels=list(strategy_counts.keys()),
        autopct="%1.1f%%",
        startangle=90,
    )
    ax3.set_title("Containers by Rule-Selected Strategy")

    ax4 = axes[1, 0]
    cluster_colors = plt.get_cmap("tab10")
    pair_labels = sorted(
        {(assignment.crane_id, assignment.truck_id) for assignment in schedule}
    )
    pair_positions = {pair: index for index, pair in enumerate(pair_labels)}
    for assignment in schedule:
        y_position = pair_positions[(assignment.crane_id, assignment.truck_id)]
        duration = assignment.completion_time - assignment.start_time
        ax4.barh(
            y_position,
            duration,
            left=assignment.start_time,
            height=0.75,
            color=cluster_colors(assignment.cluster_id % 10),
            alpha=0.85,
        )
        ax4.text(
            assignment.start_time + duration / 2,
            y_position,
            f"C{assignment.container_id}",
            ha="center",
            va="center",
            fontsize=8,
            color="white",
        )
    ax4.set_yticks(range(len(pair_labels)))
    ax4.set_yticklabels([f"QC{crane + 1}-T{truck + 1}" for crane, truck in pair_labels])
    ax4.set_xlabel("Time (minutes)")
    ax4.set_ylabel("Crane–truck pair")
    ax4.set_title("Assignments (Color Indicates Cluster)")
    ax4.grid(True, alpha=0.3)
    cluster_handles = [
        plt.Line2D(
            [0], [0], color=cluster_colors(cluster_id % 10), lw=6,
            label=f"Cluster {cluster_id}",
        )
        for cluster_id in sorted(profiles)
    ]
    ax4.legend(handles=cluster_handles, title="Cluster", loc="best", fontsize=8)

    ax5 = axes[1, 1]
    strategy_durations: Dict[str, List[float]] = {}
    for assignment in schedule:
        strategy_durations.setdefault(assignment.cluster_strategy, []).append(
            assignment.completion_time - assignment.start_time
        )
    strategy_names = [name.replace("_", " ").title() for name in strategy_durations]
    average_durations = [float(np.mean(values)) for values in strategy_durations.values()]
    bars = ax5.bar(strategy_names, average_durations, alpha=0.8)
    ax5.set_ylabel("Mean modeled assignment duration (minutes)")
    ax5.set_title("Assignment Duration by Strategy")
    ax5.tick_params(axis="x", rotation=35)
    ax5.grid(True, alpha=0.3)
    for bar, average in zip(bars, average_durations):
        ax5.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{average:.2f}",
            ha="center",
            va="bottom",
        )

    ax6 = axes[1, 2]
    ax6.axis("off")
    makespan = max(assignment.completion_time for assignment in schedule)
    average_processing = float(
        np.mean(
            [assignment.completion_time - assignment.start_time for assignment in schedule]
        )
    )
    summary_lines = [
        "SCHEDULE SUMMARY",
        "=================",
        f"Containers: {len(containers)}",
        f"Non-empty clusters: {len(profiles)}",
        f"Silhouette score: {scheduler.silhouette_score:.3f}",
        f"K-means inertia: {scheduler.inertia:.3f}",
        "",
        f"Pair-resource makespan: {makespan:.2f} min",
        f"Mean assignment duration: {average_processing:.2f} min",
        f"Fit-and-schedule time: {computation_time:.4f} s",
        "",
        "Cluster strategies:",
    ]
    summary_lines.extend(
        f"Cluster {cluster_id}: {profile.recommended_strategy} "
        f"({profile.container_count} containers)"
        for cluster_id, profile in sorted(profiles.items())
    )
    summary_lines.extend(
        ["", "Timing assumes independent crane–truck pair resources."]
    )
    ax6.text(
        0.04,
        0.96,
        "\n".join(summary_lines),
        transform=ax6.transAxes,
        fontsize=9,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "lightgray", "alpha": 0.6},
    )

    plt.tight_layout()
    plt.show()


visualize_clustering_results(scheduler, schedule, profiles, computation_time)

In [ ]:
def analyze_feature_importance():
    """Report feature variability and plot distributions without claiming importance."""
    print("\n=== FEATURE VARIABILITY ANALYSIS ===")
    features = scheduler.extract_features(containers)
    feature_stats = {}

    for index, feature_name in enumerate(scheduler.feature_names):
        values = features[:, index]
        feature_stats[feature_name] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values)),
            "min": float(np.min(values)),
            "max": float(np.max(values)),
            "range": float(np.ptp(values)),
        }

    print("\nFeature statistics (original units):")
    print(f"{'Feature':<24}{'Mean':>10}{'Std. dev.':>12}{'Range':>12}")
    print("-" * 58)
    for feature_name, stats in feature_stats.items():
        print(
            f"{feature_name:<24}{stats['mean']:>10.2f}"
            f"{stats['std']:>12.2f}{stats['range']:>12.2f}"
        )

    # K-means has no intrinsic feature-importance ranking; these plots show distributions.
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle("Feature Distributions by Cluster", fontsize=16, fontweight="bold")
    cluster_labels = scheduler.kmeans.labels_
    colors = plt.get_cmap("tab10")

    for index, feature_name in enumerate(scheduler.feature_names):
        ax = axes[index // 3, index % 3]
        for cluster_id in sorted(profiles):
            values = features[cluster_labels == cluster_id, index]
            ax.hist(
                values,
                bins=5,
                alpha=0.55,
                label=f"Cluster {cluster_id}",
                color=colors(cluster_id % 10),
            )
        display_name = feature_name.replace("_", " ").title()
        ax.set_xlabel(display_name)
        ax.set_ylabel("Container count")
        ax.set_title(f"{display_name} Distribution")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return feature_stats


feature_stats = analyze_feature_importance()

In [ ]:
def compare_with_baseline_methods():
    """Compare the schedule with the included FCFS and randomized heuristics."""
    print("\n=== BASELINE COMPARISON ===")
    clustering_makespan = max(
        assignment.completion_time for assignment in schedule
    )
    clustering_time = computation_time

    def fcfs_baseline() -> float:
        baseline_pairs = [
            CraneTruckPair(pair.crane_id, pair.truck_id) for pair in pairs
        ]
        completion_times = []
        for container in sorted(containers, key=lambda item: item.id):
            pair = min(baseline_pairs, key=lambda item: item.available_time)
            completion_time = (
                pair.available_time
                + container.handling_time
                + container.yard_distance / 2.0
            )
            completion_times.append(completion_time)
            pair.available_time = completion_time
        return max(completion_times)

    def random_baseline() -> float:
        baseline_pairs = [
            CraneTruckPair(pair.crane_id, pair.truck_id) for pair in pairs
        ]
        shuffled_containers = list(containers)
        np.random.shuffle(shuffled_containers)
        completion_times = []
        for container in shuffled_containers:
            pair = min(baseline_pairs, key=lambda item: item.available_time)
            completion_time = (
                pair.available_time
                + container.handling_time
                + container.yard_distance / 2.0
            )
            completion_times.append(completion_time)
            pair.available_time = completion_time
        return max(completion_times)

    start_time = time.time()
    fcfs_makespan = fcfs_baseline()
    fcfs_time = time.time() - start_time

    start_time = time.time()
    random_makespan = random_baseline()
    random_time = time.time() - start_time

    comparison_data = {
        "Method": ["Clustering-augmented", "FCFS", "Random order"],
        "Makespan (minutes)": [
            clustering_makespan,
            fcfs_makespan,
            random_makespan,
        ],
        "Computation time (seconds)": [
            f"{clustering_time:.4f}",
            f"{fcfs_time:.6f}",
            f"{random_time:.6f}",
        ],
        "Change vs. FCFS": [
            f"{(fcfs_makespan - clustering_makespan) / fcfs_makespan * 100:.1f}%",
            "0.0%",
            f"{(fcfs_makespan - random_makespan) / fcfs_makespan * 100:.1f}%",
        ],
        "Description": [
            "Clustering and cluster-specific rules",
            "Container IDs in ascending order",
            "One seeded random ordering",
        ],
    }
    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    methods = comparison_data["Method"]
    makespans = comparison_data["Makespan (minutes)"]
    colors = ["#45B7D1", "#FF6B6B", "#96CEB4"]
    bars = ax1.bar(methods, makespans, color=colors, alpha=0.85)
    ax1.set_ylabel("Makespan (minutes)")
    ax1.set_title("Pair-Resource Makespan")
    ax1.tick_params(axis="x", rotation=20)
    ax1.grid(True, alpha=0.3)
    for bar, makespan in zip(bars, makespans):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{makespan:.2f}",
            ha="center",
            va="bottom",
        )

    computation_times = [clustering_time, fcfs_time, random_time]
    ax2.bar(methods, computation_times, color=colors, alpha=0.85)
    ax2.set_ylabel("Computation time (seconds; logarithmic scale)")
    ax2.set_title("Measured Runtime")
    ax2.set_yscale("log")
    ax2.tick_params(axis="x", rotation=20)
    ax2.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    return comparison_df


comparison_df = compare_with_baseline_methods()

## Method interpretation and limitations

### Why combine clustering with scheduling rules?

A single ordering rule treats every container alike. Clustering offers a way to organize containers using several measured attributes at once, after which explicit rules can give different groups different scheduling priorities. In this example, K-means supplies the grouping and the rule conditions supply the strategy choice. The strategy rules are specified by the notebook; they are not learned from historical outcomes or optimized against a scheduling objective.

### Strengths and limitations

**Potential strengths**

- Standardization makes the six clustering features comparable in the Euclidean-distance calculation.
- Cluster profiles provide a compact description of groups in the original feature units.
- The strategy-selection rules are visible and can be inspected alongside the profile values.
- The example connects unsupervised grouping to a concrete, executable scheduling heuristic.

**Limitations**

- Results depend on feature selection, scaling, the chosen cluster count, and the K-means initialization; a cluster label is not a semantic category.
- The supplied demonstration data are small. They do not establish general performance or justify the same rules for other terminals.
- A silhouette score describes separation in feature space, not operational benefit. Descriptive feature variability is not feature importance.
- A single random-order baseline is only one realization, and measured runtimes can vary with the computing environment. Neither comparison establishes statistical significance.
- The pair-resource abstraction omits shared-crane and shared-truck conflicts, among other operational constraints. The reported schedule quality is therefore conditional on the stated simplifications.

### When this approach may be useful

Clustering can be useful as an exploratory way to summarize heterogeneous container data when meaningful operational features are available and domain experts can assess the resulting groups and rules. Before operational use, the assumptions, strategy rules, and resource constraints must be validated against the intended terminal and its scheduling requirements.